In [1]:
# Save the BMR for MC simulations

In [2]:
import os
import xarray as xr
import numpy as np

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"

# Load BMR for each country (lat, lon, quantile)
bmr_file = "GBD_BMR_Country_Mask_COPD_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

In [4]:
# Choose the number of samples required for MC
n_samples = 1000

# BMR from vizhub (GBD21)
bmr_mean = BMR.sel(quantile="mean")
bmr_lower = BMR.sel(quantile="lower")
bmr_upper = BMR.sel(quantile="upper")
bmr_std = (bmr_upper - bmr_lower) / (2 * 1.96)

# WARNING: this step uses a lot of memory ~80GB
bmr_samples = np.random.normal(
    bmr_mean,
    bmr_std,
    size=(n_samples, len(BMR.lat), len(BMR.lon)))

bmr_da = xr.DataArray(
    bmr_samples,
    dims=["samples", "lat", "lon"],
    coords={"samples": np.arange(n_samples), "lat": BMR.lat, "lon": BMR.lon}
).astype("float32").chunk({"samples": 10, "lat": 180, "lon": 360})

del bmr_samples

In [5]:
# === Save file in scratch directory ~25GB ===
SAVE_DIR = "/glade/derecho/scratch/awells/air_quality/BMR/"

out_file = f"GBD_BMR_Country_Mask_COPD_{n_samples}_samples_1990-2009.nc"
out_path = os.path.join(SAVE_DIR, out_file)
bmr_da.to_netcdf(out_path)